## 0. Configurando sessão spark

In [21]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("bronze_to_silver_aluno")
    .config(
        "spark.jars.packages",
        "com.google.cloud.spark:spark-bigquery-with-dependencies_2.13:0.44.2"
    )
    .getOrCreate()
)

# Projeto usado para faturamento das consultas
spark.conf.set("parentProject", "tech-challenge-fase-2-505123")

In [22]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 100)

## 1. Imports

In [23]:
from pyspark.sql import functions as F

## 2. Geração de parâmetros

In [24]:
par_source_project = "tech-challenge-fase-2-505123"
par_source_silver_uf = f"{par_source_project}.silver.uf"
par_source_diretorio_brasil = f"basedosdados.br_bd_diretorios_brasil.municipio"

par_source_gold_uf = f"{par_source_project}.gold.dim_uf"

## 3. Leitura dos dados da origem

In [25]:
df_scr_uf = spark.read.format("bigquery").option("table",par_source_silver_uf).load()
dir_mun = (spark.read.format("bigquery").option("table", par_source_diretorio_brasil).load())

## 4. Transformações

### 4.1. ufs existente na silver

In [26]:
universo_uf = (
    df_scr_uf.select("sigla_uf")
    .distinct()
)

In [27]:
dim_uf = (
    universo_uf
    .join(
        dir_mun.select("sigla_uf","nome_uf","nome_regiao").distinct(),
        on= "sigla_uf",
        how= "left"
    )
)

## 5. Armazenamento no BQ

In [29]:
(
    dim_uf.write.format("bigquery")
    .option("table", par_source_gold_uf)
    .option("writeMethod", "direct")
    .mode("overwrite")
    .save()
)

26/08/26 00:08:06 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                